## Part 1: Preprocessing

In [57]:
%pip install numpy
%pip install matplotlib
%pip install pandas
%pip install seaborn
%pip install scikit-learn
%pip install statsmodels
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.

Note: you may need to restart the kernel to use updated packages.

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [58]:
# Import our dependencies
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras import layers

#  Import and read the attrition data
attrition_df = pd.read_csv('https://static.bc-edx.com/ai/ail-v-1-0/m19/lms/datasets/attrition.csv')
attrition_df.head()

,Age,Attrition,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,HourlyRate,JobInvolvement,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,Sales,1,2,Life Sciences,2,94,3,...,3,1,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,Research & Development,8,1,Life Sciences,3,61,2,...,4,4,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,Research & Development,2,2,Other,4,92,2,...,3,2,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,Research & Development,3,4,Life Sciences,4,56,3,...,3,3,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,Research & Development,2,1,Medical,1,40,3,...,3,4,1,6,3,3,2,2,2,2


In [59]:
# Determine the number of unique values in each column
attrition_df.nunique()

Age                         43
Attrition                    2
BusinessTravel               3
Department                   3
DistanceFromHome            29
Education                    5
EducationField               6
EnvironmentSatisfaction      4
HourlyRate                  71
JobInvolvement               4
JobLevel                     5
JobRole                      9
JobSatisfaction              4
MaritalStatus                3
NumCompaniesWorked          10
OverTime                     2
PercentSalaryHike           15
PerformanceRating            2
RelationshipSatisfaction     4
StockOptionLevel             4
TotalWorkingYears           40
TrainingTimesLastYear        7
WorkLifeBalance              4
YearsAtCompany              37
YearsInCurrentRole          19
YearsSinceLastPromotion     16
YearsWithCurrManager        18
dtype: int64

In [60]:
# Create y_df with the Attrition and Department columns
y_df = attrition_df[['Attrition', 'Department']]
y_df.head()


,Attrition,Department
0,Yes,Sales
1,No,Research & Development
2,Yes,Research & Development
3,No,Research & Development
4,No,Research & Development


In [61]:
# Create a list of at least 10 column names to use as X data
X_columns = ['Age', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction', 'JobInvolvement',
             'JobLevel', 'JobRole', 'NumCompaniesWorked', 'TotalWorkingYears', 'YearsAtCompany']


# Create X_df using your selected columns
X_df = attrition_df[X_columns]


# Show the data types for X_df
X_df.dtypes


Age                         int64
DistanceFromHome            int64
Education                   int64
EnvironmentSatisfaction     int64
JobInvolvement              int64
JobLevel                    int64
JobRole                    object
NumCompaniesWorked          int64
TotalWorkingYears           int64
YearsAtCompany              int64
dtype: object

In [62]:
# Split the data into training and testing sets
from sklearn.model_selection import train_test_split


In [63]:
# Convert your X data to numeric data types however you see fit
X_df = X_df.apply(pd.to_numeric, errors='coerce')
# Add new code cells as necessary
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_df, y_df, test_size=0.2, random_state=42)
# Check for missing values in X_df
X_df.isnull().sum()
# Fill in missing values with the mean of each column
X_df.fillna(X_df.mean(), inplace=True)


In [64]:
# Create a StandardScaler
scaler = StandardScaler()


# Fit the StandardScaler to the training data
scaler.fit(X_train)


# Scale the training and testing data
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)


c:\Users\madio\anaconda3\Lib\site-packages\sklearn\utils\extmath.py:1137: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
c:\Users\madio\anaconda3\Lib\site-packages\sklearn\utils\extmath.py:1142: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
c:\Users\madio\anaconda3\Lib\site-packages\sklearn\utils\extmath.py:1162: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


In [65]:
from sklearn.preprocessing import OneHotEncoder

# Create a OneHotEncoder for the Department column
encoder = OneHotEncoder(sparse_output=False, drop='first')

# Fit the encoder to the training data
encoder.fit(y_train[['Department']])


# Create two new variables by applying the encoder
# to the training and testing data
y_train_encoded = encoder.transform(y_train[['Department']])
y_test_encoded = encoder.transform(y_test[['Department']])



In [66]:
# Create a OneHotEncoder for the Attrition column
encoder_attrition = OneHotEncoder(sparse_output=False, drop='first')    


# Fit the encoder to the training data
encoder_attrition.fit(y_train[['Attrition']])


# Create two new variables by applying the encoder
# to the training and testing data
y_train_encoded_attrition = encoder_attrition.transform(y_train[['Attrition']])
y_test_encoded_attrition = encoder_attrition.transform(y_test[['Attrition']])



## Part 2: Create, Compile, and Train the Model

In [67]:
# Find the number of columns in the X training data.
X_train.shape[1]


# Create the input layer
input_layer = layers.Input(shape=(X_train.shape[1],))


# Create at least two shared layers
shared_layer_1 = layers.Dense(128, activation='relu')(input_layer)
shared_layer_2 = layers.Dense(64, activation='relu')(shared_layer_1)



In [68]:
# Create a branch for Department
# with a hidden layer and an output layer
department_branch = layers.Dense(32, activation='relu')(shared_layer_2)
department_output = layers.Dense(y_train_encoded.shape[1], activation='softmax', name='department_output')(department_branch)


# Create the hidden layer
hidden_layer = layers.Dense(32, activation='relu')(shared_layer_2)


# Create the output layer
# with a hidden layer and an output layer
attrition_branch = layers.Dense(32, activation='relu')(hidden_layer)
# Create the output layer
attrition_output = layers.Dense(1, activation='sigmoid', name='attrition_output')(attrition_branch)





In [69]:
# Create a branch for Attrition
# with a hidden layer and an output layer
# with a hidden layer and an output layer
attrition_branch = layers.Dense(32, activation='relu')(hidden_layer)



# Create the hidden layer
hidden_layer = layers.Dense(32, activation='relu')(shared_layer_2)


# Create the output layer
# with a hidden layer and an output layer
attrition_branch = layers.Dense(32, activation='relu')(hidden_layer)



In [70]:
# Create the model
model = Model(inputs=input_layer, outputs=[department_output, attrition_output])



# Compile the model
model.compile(optimizer='adam',
              loss={'department_output': 'categorical_crossentropy', 'attrition_output': 'binary_crossentropy'},
              metrics={'department_output': 'accuracy', 'attrition_output': 'accuracy'})




# Summarize the model
model.summary()


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 10)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_20 (Dense)    │ (None, 128)       │      1,408 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_21 (Dense)    │ (None, 64)        │      8,256 │ dense_20[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_23 (Dense)    │ (None, 32)        │      2,080 │ dense_21[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_22 (Dense)    │ (None, 32)        │      2,080 │ dense_21[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_24 (Dense)    │ (None, 32)        │      1,056 │ dense_23[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ department_output   │ (None, 2)         │         66 │ dense_22[0][0]    │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attrition_output    │ (None, 1)         │         33 │ dense_24[0][0]    │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 14,979 (58.51 KB)

 Trainable params: 14,979 (58.51 KB)

 Non-trainable params: 0 (0.00 B)

In [71]:
# Ensure y_train_encoded_attrition has the correct shape

# The attrition_output layer expects the same number of columns as its units
y_train_encoded_attrition = encoder_attrition.transform(y_train[['Attrition']])
# Ensure y_train_encoded has the correct shape
y_train_encoded = encoder.transform(y_train[['Department']])

# Check the shapes of the output data
print(f"Shape of y_train_encoded: {y_train_encoded.shape}")
print(f"Shape of y_train_encoded_attrition: {y_train_encoded_attrition.shape}")

# Train the model
history = model.fit(X_train_scaled, 
                    {'department_output': y_train_encoded, 'attrition_output': y_train_encoded_attrition},
                    epochs=10,
                    batch_size=32,
                    validation_split=0.2)  # Use 20% of the training data for validation


Shape of y_train_encoded: (1176, 2)
Shape of y_train_encoded_attrition: (1176, 1)
Epoch 1/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - attrition_output_accuracy: 0.8512 - attrition_output_loss: 0.6907 - department_output_accuracy: 0.6837 - department_output_loss: 0.6565 - loss: 1.3471 - val_attrition_output_accuracy: 0.7966 - val_attrition_output_loss: 0.6843 - val_department_output_accuracy: 0.6737 - val_department_output_loss: 0.6575 - val_loss: 1.3403
Epoch 2/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - attrition_output_accuracy: 0.8298 - attrition_output_loss: 0.6813 - department_output_accuracy: 0.7032 - department_output_loss: 0.6538 - loss: 1.3350 - val_attrition_output_accuracy: 0.7966 - val_attrition_output_loss: 0.6757 - val_department_output_accuracy: 0.6737 - val_department_output_loss: 0.6501 - val_loss: 1.3249
Epoch 3/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - attrition_output_accuracy: 0.8518 - attrition_output_loss: 0.6707 - department_output_accuracy: 0.7039 - departme

In [74]:
# Evaluate the model with the testing data
model.evaluate(X_test_scaled, 
               {'department_output': y_test_encoded, 'attrition_output': y_test_encoded_attrition})
# Plot the training and validation loss and accuracy
import matplotlib.pyplot as plt
import seaborn as sns


10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - attrition_output_accuracy: 0.8447 - attrition_output_loss: 0.6059 - department_output_accuracy: 0.6917 - department_output_loss: 0.6138 - loss: 1.2197 


In [75]:
# Print the accuracy for both department and attrition
# Get the accuracy for department and attrition
department_accuracy = history.history['department_output_accuracy'][-1]


# Summary

In the provided space below, briefly answer the following questions.

1. Is accuracy the best metric to use on this data? Why or why not?

2. What activation functions did you choose for your output layers, and why?

3. Can you name a few ways that this model might be improved?

YOUR ANSWERS HERE

1. Accuracy is helpful, but not always the best for this kind of data. In the “Attrition” column (people who leave vs. stay), most people probably stay. So, a model could say "nobody leaves" and still be mostly accurate, but that’s not helpful if we really want to know who might leave. We should also check how well the model finds those who actually leave, using things like precision (how many it got right) and recall (how many it found out of all the real leavers).

2. Activation functions are like the way a model makes its final decision. For Department, we used softmax whish is good for picking one out of many options. So if the department could be Sales, HR, or Tech, softmax helps the model say: “This one looks most likely.” For Attrition, we used sigmoid, which is best when the answer is just Yes or No. It gives a number between 0 and 1, like saying: “There’s a 70% chance this person will leave.” Each one matches the kind of output we want from the model.

3. We could: 

- Make sure all text columns are turned into numbers properly. 

- Try different model sizes, like changing how many neurons (nodes) are in each layer.

- Add some layers that help the model avoid "memorizing" too much. These are called things like Dropout, and they help the model stay flexible.

- If one answer (like “No” for Attrition) happens much more often, you can tell the model to pay more attention to the rare cases (the “Yes” leavers).

- Instead of just checking accuracy, you can check how well the model finds the employees that actually leave, that's more useful in real life.